# Hidden Heterogeneity
Data is sourced from Chernozhukov et al. (2022) which adapts the approach from Cook & Ludwig (2006).

Code for preprocessing and analysis for the average treatment effect are replicated from Chernozhukov et al. (2022) Notebook 9.6.1 available at: "https://colab.research.google.com/github/CausalAIBook/MetricsMLNotebooks/blob/main/PM4/python_dml_inference_for_gun_ownership.ipynb".


## Setup
Replicates steps from Chernozhukov et al. (2022).

In [1]:
# Required package imports
import numpy as np
import pandas as pd
import re
import warnings

import statsmodels.api as sm
import statsmodels.formula.api as smf

from sklearn.model_selection import cross_val_predict, KFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import GradientBoostingRegressor

from econml.grf import CausalForest

warnings.simplefilter('ignore')
np.random.seed(42)

In [2]:
# Import data
file = "https://raw.githubusercontent.com/CausalAIBook/MetricsMLNotebooks/main/data/gun_clean.csv"
data = pd.read_csv(file)

The data are population weighted. To unweight variables, they must be divided by $\sqrt{\frac{\frac1T \sum_{t} population_{j,t}}{100000}}$ which can be derived from the time effects.

In [3]:
# County fixed effects
county_vars = data.filter(like='X_J')

# Time variables and population weights
# Pull out time variables
time_vars = data.filter(like='X_T')

# Use these to construct population weights
popweights = time_vars.sum(axis=1)

# Unweighted time variables
time_vars = time_vars.div(popweights, axis=0)

# For any columns with only zeros, drop them
time_vars = time_vars.loc[:, (time_vars != 0).any(axis=0)]

# Create time index
time_ind = (time_vars * np.arange(1, 21)).sum(axis=1)

Create initial conditions, county-level averages, and time period averages.

In [4]:
# Function to find variable names
def varlist(df=None, type=["numeric", "factor", "character"], pattern="", exclude=None):
    vars = []
    if any(t in type for t in ["numeric", "factor", "character"]):
        if "numeric" in type:
            vars += df.select_dtypes(include=["number"]).columns.tolist()
        if "factor" in type:
            vars += df.select_dtypes(include=["category"]).columns.tolist()
        if "character" in type:
            vars += df.select_dtypes(include=["object"]).columns.tolist()

    if exclude:
        vars = [var for var in vars if var not in exclude]

    if pattern:
        vars = [var for var in vars if re.search(pattern, var)]

    return vars

In [5]:
# Census control variables
census = []
census_var = ["^AGE", "^BN", "^BP", "^BZ", "^ED", "^EL", "^HI", "^HS",
              "^INC", "^LF", "^LN", "^PI", "^PO", "^PP", "^PV", "^SPR", "^VS"]

for pattern in census_var:
    census += varlist(df=data, pattern=pattern)

# Other control variables
X1 = ["logrobr", "logburg", "burg_missing", "robrate_missing"]
X2 = ["newblack", "newfhh", "newmove", "newdens", "newmal"]

# "Treatment" variable
d = "logfssl"

# Outcome variable
y = "logghomr"

# New DataFrame for time index
usedata = pd.DataFrame({'time.ind': time_ind})
usedata['weights'] = popweights

varlist_all = [y, d] + X1 + X2 + census
for var in varlist_all:
    usedata[var] = data[var]

# Construct initial conditions
varlist0 = [y] + X1 + X2 + census
for var in varlist0:
    usedata[f'{var}0'] = np.kron(usedata.loc[usedata['time.ind'] == 1, var], np.ones(20))

# County means
county_vars = pd.DataFrame(county_vars)
for var in varlist_all:
    usedata[f'{var}J'] = county_vars.dot(np.linalg.pinv(county_vars).dot(usedata[var]))

# Time means
time_vars = pd.DataFrame(time_vars)
for var in varlist_all:
    usedata[f'{var}T'] = time_vars.dot(np.linalg.pinv(time_vars).dot(usedata[var]))


Create subsamples based on population averages.

Population data is not provided so levels must be reconstructed using the population weights, which can only identify the average county population across the entire time period. The 50 and 100 largest counties are created as subsamples, to replicate Cook & Ludwig (2006).

In [6]:
# add back in County Codes
usedata['CountyCode'] = data['CountyCode']

# Reconstruct using popweights
usedata['pop_avg'] = 100000*(usedata['weights']**2)

# find largest 50 and 100 counties on average over time period
top50_counties = usedata.groupby('CountyCode')['pop_avg'].first().nlargest(50).index
top100_counties = usedata.groupby('CountyCode')['pop_avg'].first().nlargest(100).index

# create subset dataframes
usedata50 = usedata[usedata['CountyCode'].isin(top50_counties)]
usedata100 = usedata[usedata['CountyCode'].isin(top100_counties)]

# drop CountyCodes
usedata = usedata.drop(columns=['CountyCode', 'pop_avg'])
usedata50 = usedata50.drop(columns=['CountyCode', 'pop_avg'])
usedata100 = usedata100.drop(columns=['CountyCode', 'pop_avg'])

## Linear Benchmark
OLS regressions with no controls, time and cross-sectional averages, and all controls, derived from Chernozhukov et al. (2022).

In [7]:
# Baseline OLS
X = sm.add_constant(usedata['logfssl'])
y = usedata['logghomr']
lm0 = sm.OLS(y, X).fit(cov_type='HC3')
vc0 = lm0.cov_params()
coef = lm0.params['logfssl']
std_err = np.sqrt(vc0.loc['logfssl', 'logfssl'])
print(f'Baseline OLS: {coef:.4f} ({std_err:.4f})')

Baseline OLS: 0.3020 (0.0126)


In [8]:
# Regression on time and cross-sectional averages
varlistX = X1 + X2 + census
varlistMeans = [d] + X1 + X2 + census

# Adding county and time means
for var in varlistX:
    varlistMeans.append(var + 'J')
    varlistMeans.append(var + 'T')

X = sm.add_constant(usedata[varlistMeans])
y = usedata['logghomr']
lmM = sm.OLS(y, X).fit(cov_type='HC3')
vcM = lmM.cov_params()
coef = lmM.params['logfssl']
std_err = np.sqrt(vcM.loc['logfssl', 'logfssl'])
print(f'OLS with Averages: {coef:.4f} ({std_err:.4f})')

OLS with Averages: 0.1678 (0.4987)


In [9]:
# Regression on all controls
# NB: variables are dropped based on collinearity
X = sm.add_constant(usedata.drop(columns=['logghomr']))
y = usedata['logghomr']
lmA = sm.OLS(y, X).fit(cov_type='HC3')
vcA = lmA.cov_params()
coef = lmA.params['logfssl']
std_err = np.sqrt(vcA.loc['logfssl', 'logfssl'])
print(f'Full OLS: {coef:.4f} ({std_err:.4f})')

Full OLS: 0.1793 (0.0652)


### Linear Heterogeneous Treatment Effects
3 variables are pre-specified for analysis: newblack, newdens and newmal.

In [10]:
# Construct pre-defined interaction terms
int_vars = ['newblack', 'newdens', 'newmal']
int_cols = [f'logfssl_x_{w}' for w in int_vars]
dfs = {'df1': usedata, 'df2': usedata100, 'df3': usedata50}
ns = {'df1': 195, 'df2': 100, 'df3': 50}

for d in dfs.values():
    for w in int_vars:
        d[f'logfssl_x_{w}'] = d['logfssl'] * d[w]

In [11]:
# set up results storage
results = []

# loop over datasets
for name, df in dfs.items():
# loop over 3 interactions
    for w in int_vars:
        col = f'logfssl_x_{w}'
        drop_cols = ['logghomr'] + [f'logfssl_x_{v}' for v in int_vars if v != w]

        X = sm.add_constant(df.drop(columns=drop_cols))
        y = df['logghomr']
        lmA = sm.OLS(y, X).fit(cov_type='HC3')
        vcA = lmA.cov_params()

        results.append({
            'sample': name,
            'n': ns[name],
            'interaction': w,
            'coef_main': lmA.params['logfssl'],
            'std_err_main': np.sqrt(vcA.loc['logfssl', 'logfssl']),
            'pval_main': lmA.pvalues['logfssl'],
            'coef_int': lmA.params[col],
            'std_err_int': np.sqrt(vcA.loc[col, col]),
            'pval_int': lmA.pvalues[col]
        })

results_df = pd.DataFrame(results)

In [12]:
# interact all simultaneously
full_results = []

for name, df in dfs.items():
        X = sm.add_constant(df.drop(columns=['logghomr']))
        y = df['logghomr']
        lmA = sm.OLS(y, X).fit(cov_type='HC3')
        vcA = lmA.cov_params()
        
        # main effect row
        full_results.append({
                'sample': name,
                'n': ns[name],
                'interaction': 'all',
                'coef_main': lmA.params['logfssl'],
                'std_err_main': np.sqrt(vcA.loc['logfssl', 'logfssl']),
                'pval_main': lmA.pvalues['logfssl'],
                'coef_int': None,
                'std_err_int': None,
                'pval_int': None
                })

        # row per interaction
        for w in int_vars:
                col = f'logfssl_x_{w}'
                full_results.append({
                        'sample': name,
                        'n': ns[name],
                        'interaction': f'all ({w})',
                        'coef_main': lmA.params['logfssl'],
                        'std_err_main': np.sqrt(vcA.loc['logfssl', 'logfssl']),
                        'pval_main': lmA.pvalues['logfssl'],
                        'coef_int': lmA.params[col],
                        'std_err_int': np.sqrt(vcA.loc[col, col]),
                        'pval_int': lmA.pvalues[col],
                })

full_df = pd.DataFrame(full_results)
results_df = pd.concat([results_df, full_df], ignore_index=True)

## Partial Linear Model
Estimates a partial linear model using Boosted Trees, identified to be the best model by  Chernozhukov et al. (2022).

In [13]:
# model function from Chernozhukov et al. (2022)
def dml(X, D, y, modely, modeld, *, nfolds, classifier=False):
    '''
    DML for the Partially Linear Model setting with cross-fitting

    Input
    -----
    X: the controls
    D: the treatment
    y: the outcome
    modely: the ML model for predicting the outcome y
    modeld: the ML model for predicting the treatment D
    nfolds: the number of folds in cross-fitting
    classifier: bool, whether the modeld is a classifier or a regressor

    time: array of time indices, eg [0,1,...,T-1,0,1,...,T-1,...,0,1,...,T-1]
    clu: array of cluster indices, eg [1073, 1073, 1073, ..., 5055, 5055, 5055, 5055]
    cluster: bool, whether to use clustered standard errors

    Output
    ------
    point: the point estimate of the treatment effect of D on y
    stderr: the standard error of the treatment effect
    yhat: the cross-fitted predictions for the outcome y
    Dhat: the cross-fitted predictions for the treatment D
    resy: the outcome residuals
    resD: the treatment residuals
    epsilon: the final residual-on-residual OLS regression residual
    '''
    cv = KFold(n_splits=nfolds, shuffle=True, random_state=123)  # shuffled k-folds
    yhat = cross_val_predict(modely, X, y, cv=cv, n_jobs=-1)  # out-of-fold predictions for y
    # out-of-fold predictions for D
    # use predict or predict_proba dependent on classifier or regressor for D
    if classifier:
        Dhat = cross_val_predict(modeld, X, D, cv=cv, method='predict_proba', n_jobs=-1)[:, 1]
    else:
        Dhat = cross_val_predict(modeld, X, D, cv=cv, n_jobs=-1)
    # calculate outcome and treatment residuals
    resy = y - yhat
    resD = D - Dhat

    dml_data = pd.concat([pd.Series(resy, name='resy'), pd.Series(resD, name='resD')], axis=1)
    ols_mod = smf.ols(formula='resy ~ 1 + resD', data=dml_data).fit()

    point = ols_mod.params['resD']
    stderr = ols_mod.bse['resD']
    epsilon = ols_mod.resid

    return point, stderr, yhat, Dhat, resy, resD, epsilon

In [14]:
# Controls - means, initial conditions, time index
# Exclude raw dummies, weights, interaction terms
exclude_Z = (
    ['logghomr', 'logfssl', 'weights', 'time.ind'] +
    list(data.filter(like='X_J').columns) +
    list(data.filter(like='X_T').columns) +
    [f'logfssl_x_{w}' for w in int_vars]  # exclude interactions from Z
)

plm_results = []

# loop over datasets
for name, df in dfs.items():
    # define variables
    Y = df['logghomr'].to_numpy()
    D = df['logfssl'].to_numpy()
    Z_cols = [c for c in df.columns if c not in exclude_Z]
    Z = df[Z_cols].to_numpy()

    modely = make_pipeline(StandardScaler(), GradientBoostingRegressor(max_depth=4, n_iter_no_change=5))
    modeld = make_pipeline(StandardScaler(), GradientBoostingRegressor(max_depth=4, n_iter_no_change=5))
    result_BT = dml(Z, D, Y, modely, modeld, nfolds=5, classifier=False)
    point, stderr, yhat, Dhat, resy, resD, epsilon = result_BT

    # Second stage to obtain HTEs
    # Only consider newmal
    W = df['newmal'].to_numpy()

    # Residualise W on Z
    modelw = make_pipeline(StandardScaler(), GradientBoostingRegressor(max_depth=4, n_iter_no_change=5))
    cv = KFold(n_splits=5, shuffle=True, random_state=123)
    What = cross_val_predict(modelw, Z, W, cv=cv, n_jobs=-1)
    resW = W - What

    # Second stage: regress resy on resD and resD*resW
    dml_cate_data = pd.DataFrame({
        'resy': resy,
        'resD': resD,
        'resD_x_resW': resD * resW
    })

    cate_mod = smf.ols('resy ~ 1 + resD + resD_x_resW', data=dml_cate_data).fit(cov_type='HC3')

    plm_results.append({
        'sample': name,
        'n': ns[name],
        'interaction': 'newmal',
        'coef_main': cate_mod.params['resD'],
        'std_err_main': cate_mod.bse['resD'],
        'pval_main': cate_mod.pvalues['resD'],
        'coef_int': cate_mod.params['resD_x_resW'],
        'std_err_int': cate_mod.bse['resD_x_resW'],
        'pval_int': cate_mod.pvalues['resD_x_resW']
        })

plm_df = pd.DataFrame(plm_results)

## Non-Parametric Model
Use Causal Forests to estimate a fully non-parametric model.

In [15]:
cf_results = []
tau_hats = {}
importance_dfs = []

# loop over datasets
for name, df in dfs.items():
    # define variables
    Y = df['logghomr'].to_numpy()
    D = df['logfssl'].to_numpy()
    Z_cols = [c for c in df.columns if c not in exclude_Z]
    Z = df[Z_cols].to_numpy()

    cf = CausalForest(
        n_estimators=2000,
        min_samples_leaf=5,
        max_features=0.5,
        inference=True,
        random_state=123)

    cf.fit(Z, D, Y)

    # Get estimates and confidence intervals
    tau_hat = cf.oob_predict(Z).flatten()          # point estimates
    tau_lower, tau_upper = cf.predict_interval(Z, alpha=0.05)  # 95% CI
    tau_lower = tau_lower.flatten()
    tau_upper = tau_upper.flatten()

    tau_hats[name] = {
        'tau_hat': tau_hat,
        'tau_lower': tau_lower,
        'tau_upper': tau_upper,
        'resy': Y - Y.mean(),
        'resD': D - D.mean()
    }

    cf_results.append({
        'sample': name,
        'n': ns[name],
        'tau_mean': tau_hat.mean(),
        'tau_std': tau_hat.std(),
        'tau_upper': tau_upper.mean(),
        'tau_lower': tau_lower.mean(),
        })

    # extract top 10 features
    importances = cf.feature_importances_
    importance_df = pd.DataFrame({
        'variable': Z_cols,
        'importance': importances
        }).sort_values('importance', ascending=False).head(10)
    importance_df['sample'] = name
    importance_df['n'] = ns[name]
    importance_dfs.append(importance_df)

importance_all = pd.concat(importance_dfs, ignore_index=True)
cf_df = pd.DataFrame(cf_results)


### Best Linear Prediction Test
Tests for whether the heterogeneity detected is statistically meaningful.

In [ ]:
# only using pre-specified variables
blp_summaries = {}

for name, df in dfs.items():
    blp_data = pd.DataFrame({
        'cf_cate': tau_hats[name]['tau_hat'],
        'newblack': df['newblack'].values,
        'newdens':  df['newdens'].values,
        'newmal':   df['newmal'].values
    })

    blp_mod = smf.ols('cf_cate ~ 1 + newblack + newdens + newmal', data=blp_data).fit(cov_type='HC3')
    blp_summaries[name] = {
        'table': blp_mod.summary().tables[1],
        'rsquared': blp_mod.rsquared
    }

In [ ]:
# add baseline homicide rates
blp_summaries2 = {}

for name, df in dfs.items():
    blp_data = pd.DataFrame({
        'cf_cate': tau_hats[name]['tau_hat'],
        'newblack': df['newblack'].values,
        'newdens':  df['newdens'].values,
        'newmal':   df['newmal'].values,
        'logghomrJ': df['logghomrJ'].values
    })

    blp_mod = smf.ols('cf_cate ~ 1 + newblack + newdens + newmal + logghomrJ', data=blp_data).fit(cov_type='HC3')
    blp_summaries2[name] = {
        'table': blp_mod.summary().tables[1],
        'rsquared': blp_mod.rsquared
    }

In [55]:
# check correlations between variables
corrs = {}

for name, df in dfs.items():
    corr = df[['newblack', 'newdens', 'newmal', 'logghomrJ']].corr()
    corrs[name] = {
        'corr': pd.DataFrame(corr)
    }

In [56]:
# univariate regressions
blp_unis = {}

for name, df in dfs.items():
    blp_data = pd.DataFrame({
        'cf_cate': tau_hats[name]['tau_hat'],
        'newblack': df['newblack'].values,
        'newdens':  df['newdens'].values,
        'newmal':   df['newmal'].values,
        'logghomrJ': df['logghomrJ'].values
    })

    blp_uni1 = smf.ols('cf_cate ~ 1 + newblack', data=blp_data).fit(cov_type='HC3')
    blp_uni2 = smf.ols('cf_cate ~ 1 + newdens', data=blp_data).fit(cov_type='HC3')
    blp_uni3 = smf.ols('cf_cate ~ 1 + newmal', data=blp_data).fit(cov_type='HC3')
    blp_uni4 = smf.ols('cf_cate ~ 1 + logghomrJ', data=blp_data).fit(cov_type='HC3')


    blp_unis[name] = {
        'table_newblack': blp_uni1.summary().tables[1],
        'table_newdens': blp_uni2.summary().tables[1],
        'table_newmal': blp_uni3.summary().tables[1],
        'table_logghomrJ': blp_uni4.summary().tables[1]
    }